In [0]:
from pyspark.sql.functions import (
    col, trim, upper, split, explode, regexp_extract, when, lit, length,
    monotonically_increasing_id, expr, count, size, lower
)

# Read Chicago Bronze
chicago_bronze = spark.table("workspace.default.bronze_chicago_inspections")
print(f"Bronze rows: {chicago_bronze.count()}")

In [0]:
# ============================================================================
# SILVER LAYER VALIDATION RULES (Chicago)
# Per assignment: drop rows that fail these expectations
# ============================================================================

chicago_validated = (
    chicago_bronze
    # Rule 1: DBA Name (restaurant name) cannot be null
    .where(col("dba_name").isNotNull() & (length(trim(col("dba_name"))) > 0))
    # Rule 2: Inspection Date cannot be null
    .where(col("inspection_date").isNotNull())
    # Rule 3: Inspection Type cannot be null
    .where(col("inspection_type").isNotNull() & (length(trim(col("inspection_type"))) > 0))
    # Rule 4: Zip code cannot be null and must be valid 5-digit format
    .where(col("zip").isNotNull())
    .withColumn("zip_str", col("zip").cast("string"))
    .where(col("zip_str").rlike("^\\d{5}$"))
    # Rule 6: Chicago inspection results cannot be null
    .where(col("results").isNotNull() & (length(trim(col("results"))) > 0))
)

dropped = chicago_bronze.count() - chicago_validated.count()
print(f"Rows after validation: {chicago_validated.count()} (dropped {dropped})")

In [0]:
# ============================================================================
# DERIVE VIOLATION SCORE FOR CHICAGO
# Per assignment: Pass=90, Pass w/ Conditions=80, Fail=70, No Entry=0, Others=NULL
# ============================================================================

chicago_scored = chicago_validated.withColumn(
    "inspection_score",
    when(col("results") == "Pass", 90)
    .when(col("results") == "Pass w/ Conditions", 80)
    .when(col("results") == "Fail", 70)
    .when(col("results") == "No Entry", 0)
    .otherwise(lit(None).cast("int"))
)

# Verify the mapping
print("=== SCORE DERIVATION CHECK ===")
display(chicago_scored.groupBy("results", "inspection_score").count().orderBy("results"))

In [0]:
# ============================================================================
# PARSE CHICAGO VIOLATIONS
# Format: "CODE. DESCRIPTION - Comments: INSPECTOR_NOTES | CODE. DESC - Comments: NOTES | ..."
# 
# Strategy:
#   1. Split on " | " to get individual violation strings
#   2. Explode into one row per violation per inspection
#   3. Regex extract: violation_code, violation_description, violation_comments
# ============================================================================

# Split violations into rows — keep inspections with no violations as-is
chicago_with_violations = chicago_scored.where(col("violations").isNotNull() & (length(trim(col("violations"))) > 0))
chicago_no_violations = chicago_scored.where(col("violations").isNull() | (length(trim(col("violations"))) == 0))

# Parse violations into individual rows
chicago_parsed = (
    chicago_with_violations
    .withColumn("violation_raw", explode(split(col("violations"), "\\|")))
    .withColumn("violation_raw", trim(col("violation_raw")))
    # Filter out empty strings from split
    .where(length(col("violation_raw")) > 0)
    # Extract violation code (leading number before the period)
    .withColumn("violation_code", regexp_extract(col("violation_raw"), r"^(\d+)\.", 1))
    # Extract description (text between code and "- Comments:")
    .withColumn("violation_description", trim(regexp_extract(col("violation_raw"), r"^\d+\.\s*(.+?)\s*-\s*Comments:", 1)))
    # Extract comments (everything after "- Comments:")
    .withColumn("violation_comments", trim(regexp_extract(col("violation_raw"), r"-\s*Comments:\s*(.*)", 1)))
)

print(f"Inspections with violations: {chicago_with_violations.count()}")
print(f"Inspections without violations: {chicago_no_violations.count()}")
print(f"Total violation rows after parsing: {chicago_parsed.count()}")

print("\n=== SAMPLE PARSED VIOLATIONS ===")
display(chicago_parsed.select("inspection_id", "violation_code", "violation_description", "violation_comments").limit(10))

In [0]:
# ============================================================================
# Rule 7: Every inspection must have at least 1 unique violation
# Duplicate violations within the same inspection should be loaded as distinct
# ============================================================================

chicago_deduped = chicago_parsed.dropDuplicates(
    ["inspection_id", "violation_code", "violation_description"]
)

print(f"Before dedup: {chicago_parsed.count()}")
print(f"After dedup: {chicago_deduped.count()}")
print(f"Duplicates removed: {chicago_parsed.count() - chicago_deduped.count()}")

In [0]:
# ============================================================================
# BUILD CHICAGO SILVER TABLE
# Includes both violation rows (parsed) AND no-violation inspections (NULL violation fields)
# ============================================================================

# Common column selections shared by both paths
def standardize_chicago(df):
    return df.select(
        lit("Chicago").alias("source_city"),
        col("inspection_id").cast("string").alias("inspection_id"),
        col("inspection_date"),
        upper(trim(col("dba_name"))).alias("dba_name"),
        upper(trim(col("aka_name"))).alias("aka_name"),
        col("license").cast("string").alias("license_number"),
        col("facility_type"),
        col("risk").alias("risk_level"),
        upper(trim(col("address"))).alias("address"),
        upper(trim(when(col("city").isNotNull(), col("city")).otherwise(lit("CHICAGO")))).alias("city"),
        upper(trim(when(col("state").isNotNull(), col("state")).otherwise(lit("IL")))).alias("state"),
        col("zip_str").alias("zip_code"),
        col("latitude"),
        col("longitude"),
        trim(col("inspection_type")).alias("inspection_type"),
        trim(col("results")).alias("results"),
        col("inspection_score"),
        col("violation_code"),
        col("violation_description"),
        col("violation_detail"),
        col("violation_points"),
        col("violation_slot"),
        col("violation_severity")
    )

# Violation rows — already parsed and deduped
violation_rows = (
    chicago_deduped
    .withColumn("violation_description", trim(col("violation_description")))
    .withColumn("violation_detail", trim(col("violation_comments")))
    .withColumn("violation_points", lit(None).cast("int"))
    .withColumn("violation_slot", lit(None).cast("int"))
    .withColumn("violation_severity",
        when(lower(col("violation_raw")).rlike("(urgent|critical|priority|imminent)"), "Critical")
        .when(lower(col("violation_raw")).rlike("(serious|major)"), "Serious")
        .otherwise("Minor"))
)

# No-violation rows — add NULL violation columns to match schema
no_violation_rows = (
    chicago_no_violations
    .withColumn("zip_str", col("zip").cast("string"))
    .withColumn("violation_code", lit(None).cast("string"))
    .withColumn("violation_description", lit(None).cast("string"))
    .withColumn("violation_detail", lit(None).cast("string"))
    .withColumn("violation_points", lit(None).cast("int"))
    .withColumn("violation_slot", lit(None).cast("int"))
    .withColumn("violation_severity", lit(None).cast("string"))
)

# Union and write
chicago_silver_full = standardize_chicago(violation_rows).union(standardize_chicago(no_violation_rows))
chicago_silver_full.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.default.silver_chicago_inspections")

total = spark.table("workspace.default.silver_chicago_inspections").count()
print(f"Chicago Silver written: {total} rows")
print(f"  Violation rows: {violation_rows.count()}")
print(f"  No-violation rows: {no_violation_rows.count()}")

In [0]:
silver = spark.table("workspace.default.silver_chicago_inspections")

print("=== CHICAGO SILVER VALIDATION SUMMARY ===")
print(f"Total rows: {silver.count()}")
print(f"Unique inspections: {silver.select('inspection_id').distinct().count()}")
print(f"Unique violation codes: {silver.select('violation_code').distinct().count()}")
print(f"Date range: ", end="")
display(silver.selectExpr("min(inspection_date) as earliest", "max(inspection_date) as latest"))

print("\n=== NULL CHECK ON REQUIRED FIELDS ===")
for c in ["dba_name", "inspection_date", "inspection_type", "zip_code", "results"]:
    nulls = silver.where(col(c).isNull()).count()
    print(f"  {c}: {nulls} nulls {'✓' if nulls == 0 else '✗ FAIL'}")

print("\n=== SCORE DISTRIBUTION ===")
display(silver.groupBy("results", "inspection_score").count().orderBy("results"))

print("\n=== SEVERITY DISTRIBUTION ===")
display(silver.groupBy("violation_severity").count().orderBy(col("count").desc()))